# Visualització de l'atenció del decoder ViT-GPT2

Per cada token generat, mostrem **on mira** el decoder sobre els patches del ViT (cross-attention).
Això permet distingir dos tipus de fallada:
- **Visual:** el model mira la zona incorrecta de la imatge.
- **Lingüística:** el model mira bé però descriu malament.

Apliquem la visualització als pitjors casos identificats a `outputs/predictions_v3.csv`.

In [ ]:
import os, sys
sys.path.insert(0, '..')

import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
from PIL import Image
from transformers import VisionEncoderDecoderModel, ViTImageProcessor

from test import build_tokenizer, load_model

CHECKPOINT = '../checkpoints/best_model.pt'
IMAGES_DIR = '../data/flickr8k/Images'
CSV_PATH = '../outputs/predictions_v3.csv'

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device)

In [ ]:
import os
from transformers import VisionEncoderDecoderModel

tokenizer = build_tokenizer()

# Load with eager attention so output_attentions=True actually populates
# cross_attentions. SDPA (default in PyTorch>=2) is faster but silently
# returns empty attention tuples.
_MODEL_NAME = 'nlpconnect/vit-gpt2-image-captioning'
model = VisionEncoderDecoderModel.from_pretrained(_MODEL_NAME, attn_implementation='eager')
model.decoder.resize_token_embeddings(len(tokenizer))
model.config.pad_token_id           = tokenizer.pad_token_id
model.config.decoder_start_token_id = tokenizer.bos_token_id
model.config.eos_token_id           = tokenizer.eos_token_id

if os.path.exists(CHECKPOINT):
    ckpt = torch.load(CHECKPOINT, map_location='cpu')
    model.load_state_dict(ckpt['model_state_dict'], strict=False)
    print(f"Loaded checkpoint: epoch={ckpt.get('epoch')}, val_metric={ckpt.get('val_metric')}")
else:
    print(f'(no checkpoint at {CHECKPOINT}, using pretrained weights)')

model = model.eval().to(device)
processor = ViTImageProcessor.from_pretrained(_MODEL_NAME)
print('Model loaded with eager attention.')

## Funcions de visualització

El ViT-Base parteix la imatge 224×224 en patches de 16×16 → graella **14×14 = 196 patches** (+ token CLS al principi).
El decoder fa cross-attention sobre aquestes 197 posicions. Descartem el CLS i remodelem 196→(14,14).

In [ ]:
@torch.no_grad()
def generate_with_attention(image_path, max_len=20):
    """Generate caption, then re-run a forward pass to extract cross-attentions.

    Why two passes? model.generate(output_attentions=True) is brittle across
    transformers versions (cross_attentions tuple can be empty per step when
    KV cache is on). A single forward pass with the full decoder sequence is
    deterministic and gives one cross-attention tensor per decoder layer with
    shape (batch, n_heads, seq_len, encoder_seq_len).
    """
    img = Image.open(image_path).convert('RGB')
    pixel_values = processor(images=img, return_tensors='pt').pixel_values.to(device)

    # Pass 1: greedy generation
    seq = model.generate(
        pixel_values=pixel_values,
        max_length=max_len,
        num_beams=1,
        no_repeat_ngram_size=2,
        pad_token_id=tokenizer.pad_token_id,
    )[0]  # (seq_len,) including BOS

    # Pass 2: forward with the full sequence to get attentions
    dec_attn_mask = (seq.unsqueeze(0) != tokenizer.pad_token_id).long()
    out = model(
        pixel_values=pixel_values,
        decoder_input_ids=seq.unsqueeze(0),
        decoder_attention_mask=dec_attn_mask,
        output_attentions=True,
        return_dict=True,
    )
    # out.cross_attentions: tuple of length n_layers
    #   each tensor: (batch=1, n_heads, seq_len, encoder_seq_len=197)
    return img, seq, out.cross_attentions

In [ ]:
def attention_per_token(cross_attentions, layer='last'):
    """Return one (14, 14) attention map per position in the decoder sequence.

    Input: cross_attentions = tuple of length n_decoder_layers, each tensor
           shape (batch=1, n_heads, seq_len, encoder_seq_len=197).
    Output: list of (14, 14) numpy arrays of length seq_len. CLS dropped.

    The map at position i = attention used to predict token at position i+1.
    layer: 'last' | 'mean' | int
    """
    if layer == 'last':
        attn = cross_attentions[-1]
    elif layer == 'mean':
        attn = torch.stack(list(cross_attentions)).mean(0)
    else:
        attn = cross_attentions[layer]

    # attn: (1, n_heads, seq_len, 197). Mean over heads -> (1, seq_len, 197).
    a = attn.mean(dim=1).squeeze(0)  # (seq_len, 197)
    a = a[:, 1:]  # drop CLS -> (seq_len, 196)
    return [a[i].reshape(14, 14).cpu().numpy() for i in range(a.shape[0])]

In [ ]:
def visualize(image, sequence, cross_attentions, layer='last', cols=6):
    """Show original image + one heatmap overlay per generated content token.

    sequence: (seq_len,) tensor of decoder token ids starting with BOS.
    The map at position i is the cross-attention used to predict token i+1.
    """
    maps = attention_per_token(cross_attentions, layer=layer)
    pairs = []
    for i, m in enumerate(maps):
        next_pos = i + 1
        if next_pos >= len(sequence):
            continue
        tok_id = sequence[next_pos].item()
        if tok_id in (tokenizer.bos_token_id, tokenizer.eos_token_id, tokenizer.pad_token_id):
            continue
        tok_str = tokenizer.decode([tok_id]).strip()
        if not tok_str:
            continue
        pairs.append((tok_str, m))

    if not pairs:
        print('(no attention maps to display)')
        plt.figure(figsize=(3, 3))
        plt.imshow(image.resize((224, 224)))
        plt.axis('off'); plt.show()
        return

    n = len(pairs) + 1
    rows = (n + cols - 1) // cols
    fig, axes = plt.subplots(rows, cols, figsize=(2.6 * cols, 2.6 * rows))
    axes = np.array(axes).reshape(rows, cols)

    img_resized = image.resize((224, 224))
    img_arr = np.array(img_resized)

    axes[0, 0].imshow(img_arr)
    axes[0, 0].set_title('original', fontsize=10)
    axes[0, 0].axis('off')

    for k, (tok, m) in enumerate(pairs, start=1):
        r, c = divmod(k, cols)
        ax = axes[r, c]
        m_norm = (m - m.min()) / (m.max() - m.min() + 1e-8)
        heat = Image.fromarray((m_norm * 255).astype(np.uint8)).resize((224, 224), Image.BILINEAR)
        ax.imshow(img_arr)
        ax.imshow(np.array(heat), cmap='jet', alpha=0.5)
        ax.set_title(tok, fontsize=10)
        ax.axis('off')

    for k in range(len(pairs) + 1, rows * cols):
        r, c = divmod(k, cols)
        axes[r, c].axis('off')

    plt.tight_layout()
    plt.show()

## Aplicació als pitjors casos

Seleccionem els 4 pitjors per BLEU-1 i els 2 pitjors per CLIPScore.

In [ ]:
df = pd.read_csv(CSV_PATH)
worst_bleu = df.sort_values('bleu_1').drop_duplicates('image_name').head(4)
worst_clip = df.sort_values('clip_score').drop_duplicates('image_name').head(2)
targets = pd.concat([worst_bleu, worst_clip]).drop_duplicates('image_name')
targets[['image_name', 'predicted_caption', 'bleu_1', 'clip_score']]

In [ ]:
for _, row in targets.iterrows():
    print('=' * 80)
    print(f"Image    : {row['image_name']}")
    print(f"Pred     : {row['predicted_caption']}")
    print(f"BLEU-1   : {row['bleu_1']:.3f}  |  CLIP: {row['clip_score']:.2f}")
    print(f"Refs     : {row['references'][:200]}")
    image_path = os.path.join(IMAGES_DIR, row['image_name'])
    img, seq, cross_attns = generate_with_attention(image_path)
    pred = tokenizer.decode(seq, skip_special_tokens=True)
    print(f"Pred(viz): {pred}")
    visualize(img, seq, cross_attns, layer='last')

## Cas comparatiu: un dels millors

Per contrastar, visualitzem l'atenció en una de les millors prediccions per veure com hauria de ser quan funciona.

In [ ]:
best = df.sort_values('bleu_1', ascending=False).drop_duplicates('image_name').head(2)
for _, row in best.iterrows():
    print('=' * 80)
    print(f"Image    : {row['image_name']}")
    print(f"Pred     : {row['predicted_caption']}")
    print(f"BLEU-1   : {row['bleu_1']:.3f}  |  CLIP: {row['clip_score']:.2f}")
    image_path = os.path.join(IMAGES_DIR, row['image_name'])
    img, seq, cross_attns = generate_with_attention(image_path)
    visualize(img, seq, cross_attns, layer='last')

## Cas de gènere: dona predita com a home (pre-debias)

Sobre un cas concret on el v1 va equivocar el gènere, mira si l'atenció estava posada a la zona correcta de la persona o no.

In [ ]:
# Casos coneguts de swap dona→home (del v1)
for img_name in ['3758787457_1a903ee1e9.jpg', '3767841911_6678052eb6.jpg']:
    path = os.path.join(IMAGES_DIR, img_name)
    if not os.path.exists(path):
        print(f'(missing) {img_name}')
        continue
    print('=' * 80)
    print(f'Image: {img_name}')
    img, seq, cross_attns = generate_with_attention(path)
    pred = tokenizer.decode(seq, skip_special_tokens=True)
    print(f'Pred (v3): {pred}')
    visualize(img, seq, cross_attns, layer='last')

## Notes interpretatives

Després d'executar, busca patrons:

- **Atenció difusa (heatmap pla)** sobre tota la imatge → el decoder no està "agafat" a res concret; problema de l'encoder o de fusió.
- **Atenció concentrada però zona equivocada** → el ViT codifica bé però el decoder ha après una associació apresa ("green grass" → mira l'herba aunque la imatge sigui sobre una persona).
- **Atenció correcta amb predicció incorrecta** → problema purament lingüístic del GPT-2; típic dels casos de plantilla freqüent.
- **Tokens connectors (a, the, in)** acostumen a tenir atenció difusa — això és normal i no informatiu.